# FonoKids -- Landmarks de TODO el rostro (cara completa)

Hasta ahora el modelo de video solo miraba **la boca** (20 puntos de los labios). Este notebook saca los landmarks de **toda la cara**, para que el modelo pueda aprender tambien del movimiento de la **mandibula, mejillas (contorno de la cara), nariz, ojos y cejas**.

**Que cambia respecto a `Entrenar_FonoKids.ipynb` (extraccion solo de boca):**

| | Solo boca (actual) | Cara completa (este notebook) |
|---|---|---|
| Puntos por frame | 20 (labios, indices 48-67) | **68** (indices 0-67: contorno de la cara/mandibula, cejas, nariz, ojos, boca) |
| Referencia para normalizar | comisuras de la boca | **esquinas externas de los ojos** (puntos 36 y 45), para que la mandibula y la boca se midan contra algo que no se mueve al hablar |
| Dimension por frame | 120 | **408** (posicion 136 + velocidad 136 + aceleracion 136) |
| Carpeta de salida | `data/landmarks_npy/` | **`landmarks_cara_completa_npy/`**, junto a tu carpeta de videos (no pisa nada) |

Todo lo demas es igual: detector HRNet/FAN, deteccion de saltos raros, relleno de frames sin cara, suavizado exponencial y calculo de velocidad/aceleracion. Solo se descarta un clip si **ningun** frame detecto cara.

**Funciona en Colab y en tu PC.** Se detecta solo donde esta corriendo.

### Antes de correr
1. **Pon UNA sola ruta**: en la celda 2, `RUTA_VIDEOS` = la carpeta de tu Drive que tiene las clases con sus videos (la que contiene `blanco_correcto/clips/*.avi`). Ya viene con la de tu Drive; si es otra, cambia solo esa linea. La celda 3 te muestra cuantos videos encontro por clase, para que lo veas **antes** de esperar el procesamiento.
2. En Colab activa GPU (T4 alcanza). En tu PC usa el entorno `.venv-fonokids`.
3. Corre las celdas de arriba hacia abajo. Si se corta, vuelve a correr la celda 5: salta los clips que ya tienen `.npy`.

## 1. Entorno

In [ ]:
import os, re, sys, json, math, glob, subprocess

try:
    import google.colab
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
print('Corriendo en:', 'Google Colab' if EN_COLAB else 'tu PC (local)')

if EN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')


def instalar_si_falta(modulo, paquete):
    try:
        __import__(modulo)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])


import torch
print('CUDA disponible:', torch.cuda.is_available(), '(en Colab: Entorno de ejecucion > Cambiar tipo > GPU)')

instalar_si_falta('face_alignment', 'face-alignment')
import face_alignment
import cv2
import numpy as np

## 2. Configuracion de rutas y clases

In [ ]:
RUTA_VIDEOS = '/content/drive/MyDrive/Aplicaciones_Inteligentes/SACA LANDMARKS/datos/sesiones_continuas'


def buscar_raiz_local():
    actual = os.path.abspath(os.getcwd())
    for _ in range(5):
        if os.path.isdir(os.path.join(actual, 'data')) and os.path.isdir(os.path.join(actual, 'scripts')):
            return actual
        actual = os.path.dirname(actual)
    return r'D:\FonoKids'


if not os.path.isdir(RUTA_VIDEOS):
    RUTA_VIDEOS = os.path.join(buscar_raiz_local(), 'data', 'sesiones_continuas')
RUTA_VIDEOS = RUTA_VIDEOS.rstrip('/\\')
CARPETA_SALIDA = os.path.dirname(RUTA_VIDEOS)
assert os.path.isdir(RUTA_VIDEOS), f'No encontre {RUTA_VIDEOS}: cambia RUTA_VIDEOS por la carpeta de tu Drive que tiene los videos'
print('Videos en:     ', RUTA_VIDEOS)
print('Resultados en: ', CARPETA_SALIDA)

SOLO_CLASES = None
EXTENSIONES = ('.avi', '.webm')
MAX_CLIPS_POR_CLASE = None
MIN_FRACCION_CON_CARA = 0.5

CARPETAS_VIDEOS = [RUTA_VIDEOS]
_archivada = os.path.join(os.path.dirname(CARPETA_SALIDA), 'dataARCHIVADO')
if os.path.isdir(_archivada):
    CARPETAS_VIDEOS.append(_archivada)

LANDMARKS_OUT_DIR = os.path.join(CARPETA_SALIDA, 'landmarks_cara_completa_npy')
_viejos = [os.path.join(RUTA_VIDEOS, 'data', 'landmarks_npy'), os.path.join(CARPETA_SALIDA, 'landmarks_npy')]
LANDMARKS_VIEJOS_DIR = next((c for c in _viejos if os.path.isdir(c)), _viejos[0])
LABEL_MAP_PATH = os.path.join(LANDMARKS_OUT_DIR, 'label_map.json')
os.makedirs(LANDMARKS_OUT_DIR, exist_ok=True)

## 3. Buscar los videos y ver cuantos hay por clase

In [ ]:
def clase_del_video(ruta):
    carpeta = os.path.basename(os.path.dirname(ruta))
    if carpeta.lower() != 'clips':
        return None
    return os.path.basename(os.path.dirname(os.path.dirname(ruta)))


videos_por_clase = {}
for raiz in CARPETAS_VIDEOS:
    if not os.path.isdir(raiz):
        print(f'(no existe, se omite) {raiz}')
        continue
    for carpeta, _, archivos in os.walk(raiz):
        for f in archivos:
            if f.lower().endswith(EXTENSIONES):
                ruta = os.path.join(carpeta, f)
                clase = clase_del_video(ruta)
                if clase and (SOLO_CLASES is None or clase in SOLO_CLASES):
                    videos_por_clase.setdefault(clase, {}).setdefault(os.path.splitext(f)[0], ruta)

clases = sorted(videos_por_clase)
assert clases, f'No encontre videos {EXTENSIONES} en <clase>/clips/ dentro de {RUTA_VIDEOS}'

print(f"{'clase':34s} {'videos':>7s} {'landmarks viejos (boca)':>25s}")
total_videos, total_viejos = 0, 0
for c in clases:
    n_v = len(videos_por_clase[c])
    viejos_dir = os.path.join(LANDMARKS_VIEJOS_DIR, c)
    n_o = len([f for f in os.listdir(viejos_dir) if f.endswith('.npy')]) if os.path.isdir(viejos_dir) else 0
    total_videos += n_v; total_viejos += n_o
    marca = '' if n_o == 0 or n_v >= n_o else '   <-- faltan videos'
    print(f'{c:34s} {n_v:7d} {n_o:25d}{marca}')
print(f"{'TOTAL':34s} {total_videos:7d} {total_viejos:25d}")
print(f'{len(clases)} clases con videos.')
if total_viejos and total_videos < total_viejos:
    print(f'\nOJO: hay {total_videos} videos para {total_viejos} landmarks viejos. Con la cara completa solo se pueden '
          f'procesar los clips cuyo VIDEO original existe: sube el resto de los videos antes de continuar si quieres usar todos los datos.')

## 4. Codigo de extraccion de la cara completa

Cambios respecto al de solo boca (todo lo demas es identico):

- se usan los **68 puntos** (antes 48-67);
- se normaliza contra las **esquinas externas de los ojos** (36 y 45): centro = punto medio, escala = distancia entre ellas, rotacion = para dejarlas horizontales. Asi la mandibula y las mejillas se miden contra la parte rigida de la cara.

In [ ]:
FACE_INDICES = list(range(68))
EYE_OUTER_RIGHT = 36
EYE_OUTER_LEFT = 45

POSITION_DIM = len(FACE_INDICES) * 2
LANDMARK_DIM = POSITION_DIM * 3
SCALE_EPSILON = 1e-6
OUTLIER_MAX_JUMP = 0.15
SMOOTHING_ALPHA = 0.4
MIN_DETECTION_RATE_TRAIN = 0.6


def build_hrnet_detector():
    try:
        import torch._dynamo
        torch._dynamo.config.suppress_errors = True
    except Exception:
        pass
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    if device == 'cpu':
        print('[AVISO] No hay GPU: va a funcionar, pero mas lento.')
    detector = face_alignment.FaceAlignment(
        face_alignment.LandmarksType.TWO_D, device=device, flip_input=False, compile=False,
    )
    print(f'>>> Detector HRNet/FAN corriendo en {device.upper()}. <<<')
    return detector


def detect_hrnet(landmarker, frame_bgr):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    preds = landmarker.get_landmarks(rgb)
    if not preds:
        return None
    return preds[0].astype(np.float32)


def normalize_face_landmarks(landmarks_px):
    right_eye = landmarks_px[EYE_OUTER_RIGHT].astype(np.float64)
    left_eye = landmarks_px[EYE_OUTER_LEFT].astype(np.float64)
    center = (right_eye + left_eye) / 2.0
    eye_vector = left_eye - right_eye
    scale = np.linalg.norm(eye_vector)
    if scale < 1e-3:
        return None
    angle = math.atan2(eye_vector[1], eye_vector[0])
    cos_a, sin_a = math.cos(-angle), math.sin(-angle)
    rotation = np.array([[cos_a, -sin_a], [sin_a, cos_a]], dtype=np.float64)
    pts = landmarks_px[FACE_INDICES].astype(np.float64)
    normalized = ((pts - center) @ rotation.T) / (scale + SCALE_EPSILON)
    return normalized.astype(np.float32).flatten()


def is_outlier(vector, last_valid_vector):
    if last_valid_vector is None:
        return False
    return float(np.abs(vector - last_valid_vector).max()) > OUTLIER_MAX_JUMP


def fill_missing_frames(positions):
    validos = [i for i, p in enumerate(positions) if p is not None]
    if not validos:
        return None
    primero, ultimo = validos[0], validos[-1]
    relleno = list(positions)
    for i in range(primero):
        relleno[i] = relleno[primero]
    i = primero
    while i < ultimo:
        if relleno[i + 1] is None:
            j = i + 1
            while relleno[j] is None:
                j += 1
            inicio, fin = relleno[i], relleno[j]
            pasos = j - i
            for k in range(1, pasos):
                t = k / pasos
                relleno[i + k] = (1 - t) * inicio + t * fin
            i = j
        else:
            i += 1
    for i in range(ultimo + 1, len(relleno)):
        relleno[i] = relleno[ultimo]
    return np.stack(relleno, axis=0).astype(np.float32)


def smooth_positions(positions, alpha=SMOOTHING_ALPHA):
    smoothed = np.empty_like(positions)
    smoothed[0] = positions[0]
    for t in range(1, positions.shape[0]):
        smoothed[t] = alpha * positions[t] + (1 - alpha) * smoothed[t - 1]
    return smoothed.astype(np.float32)


def add_dynamics(positions):
    velocity = np.zeros_like(positions)
    velocity[1:] = positions[1:] - positions[:-1]
    acceleration = np.zeros_like(positions)
    acceleration[1:] = velocity[1:] - velocity[:-1]
    return np.concatenate([positions, velocity, acceleration], axis=1).astype(np.float32)


def process_clip(landmarker, video_path):
    cap = cv2.VideoCapture(video_path)
    positions, detected, total, last_valid = [], 0, 0, None
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        total += 1
        landmarks_px = detect_hrnet(landmarker, frame)
        vector = normalize_face_landmarks(landmarks_px) if landmarks_px is not None else None
        if vector is not None and is_outlier(vector, last_valid):
            vector = None
        if vector is not None:
            detected += 1
            last_valid = vector
        positions.append(vector)
    cap.release()
    filled = fill_missing_frames(positions)
    if filled is None:
        return None, detected, total
    return add_dynamics(smooth_positions(filled)), detected, total


print(f'Dimension por frame: {LANDMARK_DIM} = posicion {POSITION_DIM} + velocidad {POSITION_DIM} + aceleracion {POSITION_DIM}')

## 5. Correr la extraccion

Guarda un `.npy` por clip en `landmarks_cara_completa_npy/<clase>/` (junto a tu carpeta de videos). Si se corta, vuelve a correr esta celda: salta los que ya estan.

In [ ]:
landmarker = build_hrnet_detector()

label_map = {c: i for i, c in enumerate(clases)}
with open(LABEL_MAP_PATH, 'w', encoding='utf-8') as f:
    json.dump(label_map, f, ensure_ascii=False, indent=2)

total_ok, total_fail = 0, 0
descartados = {}
for word in clases:
    clips = sorted(videos_por_clase[word].items())
    if MAX_CLIPS_POR_CLASE:
        clips = clips[:MAX_CLIPS_POR_CLASE]
    if not clips:
        continue
    out_dir = os.path.join(LANDMARKS_OUT_DIR, word)
    os.makedirs(out_dir, exist_ok=True)
    pendientes = [(n, p) for n, p in clips if not os.path.exists(os.path.join(out_dir, n + '.npy'))]
    print(f'\n[{word}] {len(clips) - len(pendientes)} ya tenian landmarks, {len(pendientes)} por procesar.')
    for i, (nombre, ruta) in enumerate(pendientes, 1):
        sequence, detected, total = process_clip(landmarker, ruta)
        if sequence is None:
            print(f'  [{word} {i}/{len(pendientes)}] {nombre}: NINGUN frame detecto cara -- descartado.')
            total_fail += 1
            continue
        if total and detected / total < MIN_FRACCION_CON_CARA:
            descartados.setdefault(word, []).append(nombre)
            continue
        np.save(os.path.join(out_dir, nombre + '.npy'), sequence)
        total_ok += 1
        if i % 25 == 0 or i == len(pendientes):
            print(f'  [{word} {i}/{len(pendientes)}] {nombre}: cara en {detected}/{total} frames -> {sequence.shape}')

print(f'\n=== TOTAL: {total_ok} guardados | {total_fail} descartados (sin cara en ningun frame) ===')
print('Landmarks de cara completa en:', LANDMARKS_OUT_DIR)
n_baja = sum(len(v) for v in descartados.values())
print(f'Ademas {n_baja} clips se descartaron: la cara completa se veia en menos del {MIN_FRACCION_CON_CARA*100:.0f}% de sus frames (camara que corta la cara / no se ven los ojos).')
for c, lst in sorted(descartados.items(), key=lambda kv: -len(kv[1])):
    print(f'   {c}: {len(lst)}')
with open(os.path.join(LANDMARKS_OUT_DIR, 'descartados_sin_cara_completa.json'), 'w', encoding='utf-8') as f:
    json.dump(descartados, f, ensure_ascii=False, indent=1)

## 6. Comprobacion visual

Dibuja la cara normalizada de un clip (frame del medio) para confirmar que se guardaron los 68 puntos y que la boca, mandibula, nariz y ojos quedaron donde corresponde.

In [ ]:
import matplotlib.pyplot as plt

archivos = sorted(glob.glob(os.path.join(LANDMARKS_OUT_DIR, '*', '*.npy')))
print(f'{len(archivos)} clips con landmarks de cara completa')
if archivos:
    secuencia = np.load(archivos[len(archivos) // 2])
    print('Archivo:', os.path.relpath(archivos[len(archivos) // 2], LANDMARKS_OUT_DIR), '| forma:', secuencia.shape)
    assert secuencia.shape[1] == LANDMARK_DIM, f'dimension inesperada {secuencia.shape[1]}'
    pts = secuencia[secuencia.shape[0] // 2, :POSITION_DIM].reshape(68, 2)
    grupos = {'contorno / mandibula': range(0, 17), 'cejas': range(17, 27), 'nariz': range(27, 36), 'ojos': range(36, 48), 'boca': range(48, 68)}
    plt.figure(figsize=(5, 6))
    for nombre, idx in grupos.items():
        idx = list(idx)
        plt.scatter(pts[idx, 0], -pts[idx, 1], s=18, label=nombre)
    plt.legend(fontsize=8); plt.axis('equal'); plt.title('Cara normalizada (frame del medio)')
    plt.show()

## Listo

Quedaron en `landmarks_cara_completa_npy/<clase>/*.npy` (68 puntos, 408 numeros por frame) con su `label_map.json`, junto a tu carpeta de videos.

Siguiente paso: `Entrenar_Cara_Completa_FonoKids.ipynb` (pega la MISMA ruta en `RUTA_VIDEOS`). Los audios y sus embeddings no cambian, no hace falta volver a sacarlos.